In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("hw8.ipynb")

# CPSC 330 - Applied Machine Learning

## Homework 8: Introduction to Computer vision and Time Series

**Due date: See [deliverable due dates](https://ubc-cs.github.io/cpsc330-2025W2/#deliverable-due-dates-tentative)**.

## Imports

In [2]:
from hashlib import sha1

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## Instructions
rubric={points}

You will earn points for following these instructions and successfully submitting your work on Gradescope.  

### Group wotk instructions

**You may work with a partner on this homework and submit your assignment as a group.** Below are some instructions on working as a group.  
- The maximum group size is 2.
  
- Use group work as an opportunity to collaborate and learn new things from each other. 
- Be respectful to each other and make sure you understand all the concepts in the assignment well. 
- It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. 
- You can find the instructions on how to do group submission on Gradescope [here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members).
- If you would like to use late tokens for the homework, all group members must have the necessary late tokens available. Please note that the late tokens will be counted for all members of the group.   


### General submission instructions

- Please **read carefully
[Use of Generative AI policy](https://ubc-cs.github.io/cpsc330-2025W2/syllabus#use-of-generative-ai-in-the-course)** before starting the homework assignment. 
- **Run all cells before submitting:** Go to `Kernel -> Restart Kernel and Clear All Outputs`, then select `Run -> Run All Cells`. This ensures your notebook runs cleanly from start to finish without errors.
  
- **Submit your files on Gradescope.**  
   - Upload only your `.ipynb` file **with outputs displayed** and any required output files.
     
   - Do **not** submit other files from your repository.  
   - If you need help, see the [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/).  
- **Check that outputs render properly.**  
   - Make sure all plots and outputs appear in your submission.
     
   - If your `.ipynb` file is too large and doesn't render on Gradescope, also upload a PDF or HTML version so the TAs can view your work.  
- **Keep execution order clean.**  
   - Execution numbers must start at "1" and increase in order.
     
   - Notebooks without visible outputs may not be graded.  
   - Out-of-order or missing execution numbers may result in mark deductions.  
- **Follow course submission guidelines:** Review the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions) for detailed guidance on completing and submitting assignments. 
   
</div>

_Points:_ 2

<!-- END QUESTION -->

<br><br>

## Exercise 1: time series prediction

In this exercise we'll be looking at a [dataset of avocado prices](https://www.kaggle.com/neuromusic/avocado-prices). You should start by downloading the dataset and storing it under the `data` folder. We will be forcasting average avocado price for the next week. 

In [3]:
df = pd.read_csv("data/avocado.csv", parse_dates=["Date"], index_col=0)
df.head()

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany
1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany
2,2015-12-13,0.93,118220.22,794.70,109149.67,130.50,8145.35,8042.21,103.14,0.0,conventional,2015,Albany
3,2015-12-06,1.08,78992.15,1132.00,71976.41,72.58,5811.16,5677.40,133.76,0.0,conventional,2015,Albany
4,2015-11-29,1.28,51039.60,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany


In [4]:
df.shape

(18249, 13)

In [5]:
df["Date"].min()

Timestamp('2015-01-04 00:00:00')

In [6]:
df["Date"].max()

Timestamp('2018-03-25 00:00:00')

It looks like the data ranges from the start of 2015 to March 2018 (~2 years ago), for a total of 3.25 years or so. Let's split the data so that we have a 6 months of test data.

In [7]:
split_date = '20170925'
df_train = df[df["Date"] <= split_date]
df_test  = df[df["Date"] >  split_date]

In [8]:
assert len(df_train) + len(df_test) == len(df)

<br><br>

<!-- BEGIN QUESTION -->

### 1.1 How many time series? 
rubric={points:4}

In the [Rain in Australia](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package) dataset from lecture demo, we had different measurements for each Location. 

We want you to consider this for the avocado prices dataset. For which categorical feature(s), if any, do we have separate measurements? Justify your answer by referencing the dataset.

<div class="alert alert-warning">

Solution_1.1
    
</div>

_Points:_ 4

Both type and region are categorical features that have separate measurements. We can see this by checking the unique combinations of type and region in the dataset, which shows that there are multiple entries for each combination of type and region. Additionally, when we check for a single date, we find that there are different type-region combinations present, confirming that these features have separate measurements.

In [9]:
# Check unique values in categorical columns
print("Unique types:", df["type"].unique())
print("Unique regions:", df["region"].unique())
print("\nNumber of unique types:", df["type"].nunique())
print("Number of unique regions:", df["region"].nunique())

Unique types: <StringArray>
['conventional', 'organic']
Length: 2, dtype: str
Unique regions: <StringArray>
[             'Albany',             'Atlanta', 'BaltimoreWashington',
               'Boise',              'Boston',    'BuffaloRochester',
          'California',           'Charlotte',             'Chicago',
    'CincinnatiDayton',            'Columbus',       'DallasFtWorth',
              'Denver',             'Detroit',         'GrandRapids',
          'GreatLakes',  'HarrisburgScranton', 'HartfordSpringfield',
             'Houston',        'Indianapolis',        'Jacksonville',
            'LasVegas',          'LosAngeles',          'Louisville',
   'MiamiFtLauderdale',            'Midsouth',           'Nashville',
    'NewOrleansMobile',             'NewYork',           'Northeast',
  'NorthernNewEngland',             'Orlando',        'Philadelphia',
       'PhoenixTucson',          'Pittsburgh',              'Plains',
            'Portland',   'RaleighGreensboro',     '

In [10]:
# Check combinations of type and region
print("\nUnique (type, region) combinations:", df.groupby(["type", "region"]).ngroups)


Unique (type, region) combinations: 108


In [11]:
# See unique combinations
print(df[["Date", "type", "region", "AveragePrice"]].drop_duplicates(subset=["type", "region"]).sort_values("region"))

         Date          type               region  AveragePrice
0  2015-12-27  conventional               Albany          1.33
0  2015-12-27       organic               Albany          1.83
0  2015-12-27  conventional              Atlanta          0.99
0  2015-12-27       organic              Atlanta          1.84
0  2015-12-27  conventional  BaltimoreWashington          1.17
..        ...           ...                  ...           ...
0  2015-12-27  conventional              TotalUS          0.95
0  2015-12-27       organic                 West          1.46
0  2015-12-27  conventional                 West          0.83
0  2015-12-27  conventional     WestTexNewMexico          0.71
0  2015-12-27       organic     WestTexNewMexico          1.81

[108 rows x 4 columns]


In [12]:
# Check if same date appears for different types/regions
sample_date = df["Date"].iloc[0]
print(f"On {sample_date}:")
print(df[df["Date"] == sample_date][["type", "region"]].value_counts())

On 2015-12-27 00:00:00:
type          region             
conventional  Albany                 1
              Atlanta                1
              BaltimoreWashington    1
              Boise                  1
              Boston                 1
                                    ..
organic       Syracuse               1
              Tampa                  1
              TotalUS                1
              West                   1
              WestTexNewMexico       1
Name: count, Length: 108, dtype: int64


In [13]:
print(df.groupby(["type", "region"]).size())

type          region             
conventional  Albany                 169
              Atlanta                169
              BaltimoreWashington    169
              Boise                  169
              Boston                 169
                                    ... 
organic       Syracuse               169
              Tampa                  169
              TotalUS                169
              West                   169
              WestTexNewMexico       166
Length: 108, dtype: int64


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.2 Equally spaced measurements? 
rubric={points:4}

In the Rain in Australia dataset, the measurements were generally equally spaced but with some exceptions. How about with this dataset? Justify your answer by referencing the dataset.

<div class="alert alert-warning">

Solution_1.2
    
</div>

_Points:_ 4

Yes, measurements are nearly equally spaced with weekly frequency. Out of 18,141 total time differences across all type-region combinations, 18,139 (99.99%) are exactly 7 days apart. Only 2 rare exceptions exist: one 14-day gap and one 21-day gap, likely due to holidays or data collection issues.

In [14]:
df_sort = df.sort_values(by=["region", "type", "Date"]).reset_index(drop=True)

# For a single type-region combination, examine the date differences
sample_combo = df_sort[(df_sort["region"] == "Albany") & (df_sort["type"] == "conventional")]
print(sample_combo[["Date", "AveragePrice"]].head(10))

        Date  AveragePrice
0 2015-01-04          1.22
1 2015-01-11          1.24
2 2015-01-18          1.17
3 2015-01-25          1.06
4 2015-02-01          0.99
5 2015-02-08          0.99
6 2015-02-15          1.06
7 2015-02-22          1.07
8 2015-03-01          0.99
9 2015-03-08          1.07


In [15]:
# Calculate time differences
date_diffs = sample_combo["Date"].diff()
print(date_diffs.value_counts())

Date
7 days    168
Name: count, dtype: int64


In [16]:
# Check for all combinations
all_diffs = []
for (region, avotype), group in df_sort.groupby(["region", "type"]):
    diffs = group["Date"].diff().dropna()
    all_diffs.extend(diffs.values)

all_diffs_days = [int(d / np.timedelta64(1, 'D')) for d in all_diffs]

from collections import Counter
diff_counts = Counter(all_diffs_days)
print(f"\nUnique time differences (in days): {sorted(diff_counts.keys())}")
for days in sorted(diff_counts.keys()):
    print(f"  {days} days: {diff_counts[days]} occurrences")

print(f"\nTotal time differences: {len(all_diffs_days)}")
print(f"Most common difference: {max(diff_counts, key=diff_counts.get)} days")


Unique time differences (in days): [7, 14, 21]
  7 days: 18139 occurrences
  14 days: 1 occurrences
  21 days: 1 occurrences

Total time differences: 18141
Most common difference: 7 days


In [17]:
# Check if any combination has irregular spacing
print(f"\nTotal time differences: {len(all_diffs_days)}")
print(f"Most common difference: {max(diff_counts, key=diff_counts.get)} days")


Total time differences: 18141
Most common difference: 7 days


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Interpreting regions 
rubric={points:4}

In the Rain in Australia dataset, each location was a different place in Australia. For this dataset, look at the names of the regions. Do you think the regions are also all distinct, or are there overlapping regions? Justify your answer by referencing the data.

<div class="alert alert-warning">

Solution_1.3
    
</div>

_Points:_ 4

Yes, regions overlap. The dataset contains both specific regions (California, Boston) and aggregate regions (West, Northeast, TotalUS) that include them. This creates hierarchical overlap unlike the Rain in Australia dataset where each location is distinct.

In [18]:
# Look at all unique regions
unique_regions = sorted(df["region"].unique())
for i, region in enumerate(unique_regions, 1):
    print(f"  {i:2d}. {region}")

print(f"\nTotal regions: {len(unique_regions)}")

   1. Albany
   2. Atlanta
   3. BaltimoreWashington
   4. Boise
   5. Boston
   6. BuffaloRochester
   7. California
   8. Charlotte
   9. Chicago
  10. CincinnatiDayton
  11. Columbus
  12. DallasFtWorth
  13. Denver
  14. Detroit
  15. GrandRapids
  16. GreatLakes
  17. HarrisburgScranton
  18. HartfordSpringfield
  19. Houston
  20. Indianapolis
  21. Jacksonville
  22. LasVegas
  23. LosAngeles
  24. Louisville
  25. MiamiFtLauderdale
  26. Midsouth
  27. Nashville
  28. NewOrleansMobile
  29. NewYork
  30. Northeast
  31. NorthernNewEngland
  32. Orlando
  33. Philadelphia
  34. PhoenixTucson
  35. Pittsburgh
  36. Plains
  37. Portland
  38. RaleighGreensboro
  39. RichmondNorfolk
  40. Roanoke
  41. Sacramento
  42. SanDiego
  43. SanFrancisco
  44. Seattle
  45. SouthCarolina
  46. SouthCentral
  47. Southeast
  48. Spokane
  49. StLouis
  50. Syracuse
  51. Tampa
  52. TotalUS
  53. West
  54. WestTexNewMexico

Total regions: 54


In [19]:
# Check for aggregate/summary regions
aggregate_keywords = ["Total", "US", "Northeast", "Southeast", "Midwest", "West", "South", "Plains", "Great", "Mid"]
for region in unique_regions:
    for keyword in aggregate_keywords:
        if keyword.lower() in region.lower():
            print(f"  {region}")
            break

  Columbus
  GreatLakes
  Houston
  Midsouth
  Northeast
  Plains
  SouthCarolina
  SouthCentral
  Southeast
  Syracuse
  TotalUS
  West
  WestTexNewMexico


In [20]:
# Check if specific cities/states are also in broader regions
print(f"  'TotalUS' in regions: {'TotalUS' in unique_regions}")
print(f"  'California' in regions: {'California' in unique_regions}")
print(f"  'West' in regions: {'West' in unique_regions}")
print(f"  'Northeast' in regions: {'Northeast' in unique_regions}")

  'TotalUS' in regions: True
  'California' in regions: True
  'West' in regions: True
  'Northeast' in regions: True


In [21]:
# Count data per region
region_counts = df["region"].value_counts().sort_values(ascending=False)
print(region_counts.head(10))

region
Albany                 338
Atlanta                338
BaltimoreWashington    338
Boise                  338
Boston                 338
BuffaloRochester       338
California             338
Charlotte              338
Chicago                338
CincinnatiDayton       338
Name: count, dtype: int64


<!-- END QUESTION -->

<br><br>

We will use the entire dataset despite any location-based weirdness uncovered in the previous part.

We will be trying to forecast the avocado price. The function below is adapted from the lecture.

In [22]:
import pandas as pd


def create_lag_feature(
    df: pd.DataFrame,
    orig_feature: str,
    lag: int,
    groupby: list[str],
    new_feature_name: str | None = None,
    clip: bool = False,
) -> pd.DataFrame:
    """
    Create a lagged (or ahead) version of a feature, optionally per group.

    Assumes df is already sorted by time within each group and has unique indices.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset.
    orig_feature : str
        Name of the column to lag.
    lag : int
        The lag:
          - negative → values from the past (t-1, t-2, ...)
          - positive → values from the future (t+1, t+2, ...)
    groupby : list of str
        Column(s) to group by if df contains multiple time series.
    new_feature_name : str, optional
        Name of the new column. If None, a name is generated automatically.
    clip : bool, default False
        If True, drop rows where the new feature is NaN.

    Returns
    -------
    pd.DataFrame
        A new dataframe with the additional column added.
    """
    if lag == 0:
        raise ValueError("lag cannot be 0 (no shift). Use the original feature instead.")

    # Default name if not provided
    if new_feature_name is None:
        if lag < 0:
            new_feature_name = f"{orig_feature}_lag{abs(lag)}"
        else:
            new_feature_name = f"{orig_feature}_ahead{lag}"

    df = df.copy()

    # Map your convention (negative=past, positive=future) to pandas shift
    # pandas: shift(+k) → past, shift(-k) → future
    periods = abs(lag) if lag < 0 else -lag

    df[new_feature_name] = (
        df.groupby(groupby, sort=False)[orig_feature]
          .shift(periods)
    )

    if clip:
        df = df.dropna(subset=[new_feature_name])

    return df


We first sort our dataframe properly:

In [23]:
df_sort = df.sort_values(by=["region", "type", "Date"]).reset_index(drop=True)
df_sort

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,2015-01-04,1.22,40873.28,2819.50,28287.42,49.90,9716.46,9186.93,529.53,0.0,conventional,2015,Albany
1,2015-01-11,1.24,41195.08,1002.85,31640.34,127.12,8424.77,8036.04,388.73,0.0,conventional,2015,Albany
2,2015-01-18,1.17,44511.28,914.14,31540.32,135.77,11921.05,11651.09,269.96,0.0,conventional,2015,Albany
3,2015-01-25,1.06,45147.50,941.38,33196.16,164.14,10845.82,10103.35,742.47,0.0,conventional,2015,Albany
4,2015-02-01,0.99,70873.60,1353.90,60017.20,179.32,9323.18,9170.82,152.36,0.0,conventional,2015,Albany
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18244,2018-02-25,1.57,18421.24,1974.26,2482.65,0.00,13964.33,13698.27,266.06,0.0,organic,2018,WestTexNewMexico
18245,2018-03-04,1.54,17393.30,1832.24,1905.57,0.00,13655.49,13401.93,253.56,0.0,organic,2018,WestTexNewMexico
18246,2018-03-11,1.56,22128.42,2162.67,3194.25,8.93,16762.57,16510.32,252.25,0.0,organic,2018,WestTexNewMexico
18247,2018-03-18,1.56,15896.38,2055.35,1499.55,0.00,12341.48,12114.81,226.67,0.0,organic,2018,WestTexNewMexico


We then call `create_lag_feature`. This creates a new column in the dataset `AveragePriceNextWeek`, which is the following week's `AveragePrice`. We have set `clip=True` which means it will remove rows where the target would be missing.

In [24]:
df_hastarget = create_lag_feature(df_sort, "AveragePrice", +1, ["region", "type"], "AveragePriceNextWeek", clip=True)
df_hastarget

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region,AveragePriceNextWeek
0,2015-01-04,1.22,40873.28,2819.50,28287.42,49.90,9716.46,9186.93,529.53,0.0,conventional,2015,Albany,1.24
1,2015-01-11,1.24,41195.08,1002.85,31640.34,127.12,8424.77,8036.04,388.73,0.0,conventional,2015,Albany,1.17
2,2015-01-18,1.17,44511.28,914.14,31540.32,135.77,11921.05,11651.09,269.96,0.0,conventional,2015,Albany,1.06
3,2015-01-25,1.06,45147.50,941.38,33196.16,164.14,10845.82,10103.35,742.47,0.0,conventional,2015,Albany,0.99
4,2015-02-01,0.99,70873.60,1353.90,60017.20,179.32,9323.18,9170.82,152.36,0.0,conventional,2015,Albany,0.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18243,2018-02-18,1.56,17597.12,1892.05,1928.36,0.00,13776.71,13553.53,223.18,0.0,organic,2018,WestTexNewMexico,1.57
18244,2018-02-25,1.57,18421.24,1974.26,2482.65,0.00,13964.33,13698.27,266.06,0.0,organic,2018,WestTexNewMexico,1.54
18245,2018-03-04,1.54,17393.30,1832.24,1905.57,0.00,13655.49,13401.93,253.56,0.0,organic,2018,WestTexNewMexico,1.56
18246,2018-03-11,1.56,22128.42,2162.67,3194.25,8.93,16762.57,16510.32,252.25,0.0,organic,2018,WestTexNewMexico,1.56


Our goal is to predict `AveragePriceNextWeek`. 

Let's split the data:

In [25]:
df_train = df_hastarget[df_hastarget["Date"] <= split_date]
df_test  = df_hastarget[df_hastarget["Date"] >  split_date]

<br><br>

<!-- BEGIN QUESTION -->

### 1.4 `AveragePrice` baseline 
rubric={points}

Soon we will want to build some models to forecast the average avocado price a week in advance. Before we start with any ML though, let's try a baseline. Previously we used `DummyClassifier` or `DummyRegressor` as a baseline. This time, we'll do something else as a baseline: we'll assume the price stays the same from this week to next week. So, we'll set our prediction of "AveragePriceNextWeek" exactly equal to "AveragePrice", assuming no change. That is kind of like saying, "If it's raining today then I'm guessing it will be raining tomorrow". This simplistic approach will not get a great score but it's a good starting point for reference. If our model does worse that this, it must not be very good. 

Using this baseline approach, what $R^2$ do you get on the train and test data?

<div class="alert alert-warning">

Solution_1.4
    
</div>

_Points:_ 4

_Type your answer here, replacing this text._

In [26]:
y_train_pred = df_train['AveragePrice']
y_train_true = df_train['AveragePriceNextWeek']
train_r2 = r2_score(y_train_true, y_train_pred)

print(f"Train R²: {train_r2:.6f}")

Train R²: 0.828580


In [27]:
y_test_pred = df_test['AveragePrice']
y_test_true = df_test['AveragePriceNextWeek']
test_r2 = r2_score(y_test_true, y_test_pred)

print(f"Test R²: {test_r2:.6f}")

Test R²: 0.763178


In [28]:
assert not train_r2 is None, "Are you using the correct variable name?"
assert not test_r2 is None, "Are you using the correct variable name?"
assert sha1(str(round(train_r2, 3)).encode('utf8')).hexdigest() == 'b1136fe2a8918904393ab6f40bfb3f38eac5fc39', "Your training score is not correct. Are you using the right features?"
assert sha1(str(round(test_r2, 3)).encode('utf8')).hexdigest() == 'cc24d9a9b567b491a56b42f7adc582f2eefa5907', "Your test score is not correct. Are you using the right features?"

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.5 Forecasting average avocado price
rubric={points:10}

Now that the baseline is done, let's build some models to forecast the average avocado price a week later. Experiment with a few approachs for encoding the date. Justify the decisions you make. Which approach worked best? Report your test score and briefly discuss your results.

Benchmark: you should be able to achieve $R^2$ of at least 0.79 on the test set. I got to 0.80, but not beyond that. Let me know if you do better!

Note: because we only have 2 splits here, we need to be a bit wary of overfitting on the test set. Try not to test on it a ridiculous number of times. If you are interested in some proper ways of dealing with this, see for example sklearn's [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html), which is like cross-validation for time series data.

<div class="alert alert-warning">

Solution_1.5
    
</div>

_Points:_ 10

I used cyclical encoding for months (sin(2π × month/12) and cos(2π × month/12)) to preserve seasonality, plus year as a linear feature. With XGBRegressor, RandomizedSearchCV, and TimeSeriesSplit, I achieved a test R² of 0.7993 (gave up trynna reach 0.80). The cyclical encoding effectively captures seasonal patterns, and TimeSeriesSplit ensures validation on future data, preventing overly optimistic estimates. The modest gap between train R² (0.87) and test R² (0.80) indicates reasonable regularization.

In [29]:
X_train = df_train.drop(columns=['AveragePriceNextWeek']).copy()
y_train = df_train['AveragePriceNextWeek']

X_test = df_test.drop(columns=['AveragePriceNextWeek']).copy()
y_test = df_test['AveragePriceNextWeek']

for df_in in [X_train, X_test]:
    df_in['Month'] = df_in['Date'].dt.month
    df_in['Year'] = df_in['Date'].dt.year
    
    # Cyclical encoding for the month
    df_in['Month_sin'] = np.sin(2 * np.pi * df_in['Month'] / 12)
    df_in['Month_cos'] = np.cos(2 * np.pi * df_in['Month'] / 12)

In [30]:
num_feats = [
    'AveragePrice', 'Total Volume', '4046', '4225', '4770', 
    'Total Bags', 'Small Bags', 'Large Bags', 'XLarge Bags', 'Year'
]
cat_feats = ['type', 'region', 'Month_sin', 'Month_cos']

In [31]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_feats),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_feats)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('xgb', XGBRegressor(random_state=123, n_jobs=-1))
])

param_grid = {
    'xgb__n_estimators': [100, 200, 300, 400],
    'xgb__learning_rate': [0.01, 0.02, 0.03, 0.05, 0.1],
    'xgb__max_depth': [4, 5, 6, 7, 8],
    'xgb__subsample': [0.7, 0.8, 0.9, 1.0],
    'xgb__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'xgb__gamma': [0, 0.1, 0.5, 1.0],
}

# Use TimeSeriesSplit for time series data
tscv = TimeSeriesSplit(n_splits=3)

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=50,
    cv=tscv,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    random_state=123
)

random_search.fit(X_train, y_train)

print(f"\nBest parameters: {random_search.best_params_}")
print(f"Best CV R² score: {random_search.best_score_:.4f}")

best_model = random_search.best_estimator_

train_score = best_model.score(X_train, y_train)
test_score = best_model.score(X_test, y_test)

print(f"\nBest Model Performance:")
print(f"Train R²: {train_score:.4f}")
print(f"Test R²:  {test_score:.4f}")

Fitting 3 folds for each of 50 candidates, totalling 150 fits

Best parameters: {'xgb__subsample': 1.0, 'xgb__n_estimators': 200, 'xgb__max_depth': 4, 'xgb__learning_rate': 0.05, 'xgb__gamma': 0, 'xgb__colsample_bytree': 0.8}
Best CV R² score: 0.8410

Best Model Performance:
Train R²: 0.8722
Test R²:  0.7993


<!-- END QUESTION -->

<br><br><br><br>

## Exercise 2: Short answer questions

<!-- BEGIN QUESTION -->

### 2.1 Time series

rubric={points:6}

The following questions pertain to Lecture 20 on time series data:

1. Sometimes a time series has missing time points or, worse, time points that are unequally spaced in general. Give an example of a real world situation where the time series data would have unequally spaced time points.
2. In class we discussed two approaches to using temporal information: encoding the date as one or more features, and creating lagged versions of features. Which of these (one/other/both/neither) two approaches would struggle with unequally spaced time points? Briefly justify your answer.
3. When studying time series modeling, we explored several ways to encode date information as a feature for the citibike dataset. When we used time of day as a numeric feature, the Ridge model was not able to capture the periodic pattern. Why? How did we tackle this problem? Briefly explain.

<div class="alert alert-warning">

Solution_2.1
    
</div>

_Points:_ 6

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.2 Computer vision 
rubric={points:6}

The following questions pertain to the lecture on multiclass classification and introduction to computer vision. 

1. How many parameters (coefficients and intercepts) will `sklearn`’s `LogisticRegression()` model learn for a four-class classification problem, assuming that you have 10 features? Briefly explain your answer.
2. In Lecture 19, we briefly discussed how neural networks are sort of like `sklearn`'s pipelines, in the sense that they involve multiple sequential transformations of the data, finally resulting in the prediction. Why was this property useful when it came to transfer learning?
3. Imagine that you have a small dataset with ~1000 images containing pictures and names of 50 different Computer Science faculty members from UBC. Your goal is to develop a reasonably accurate multi-class classification model for this task. Describe which model/technique you would use and briefly justify your choice in one to three sentences.

<div class="alert alert-warning">

Solution_2.2
    
</div>

_Points:_ 6

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br><br>

Before submitting your assignment, please make sure you have followed all the instructions in the Submission Instructions section at the top. 

Here is a quick checklist before submitting: 

- [ ] Restart kernel, clear outputs, and run all cells from top to bottom.  
- [ ] `.ipynb` file runs without errors and contains all outputs.  
- [ ] Only `.ipynb` and required output files are uploaded (no extra files).  
- [ ] Execution numbers start at **1** and are in order.  
- [ ] If `.ipynb` is too large and doesn't render on Gradescope, also upload a PDF/HTML version.  
- [ ] Reviewed the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions).  

![](img/eva-well-done.png)